# 🚀 Notebook 04 — Streamlit Application
### Senti-Recommend: Hybrid AI Tool Discovery Engine
**PGD Data Science with AI — Final Project**

---

## What This Notebook Does

This notebook:
1. **Explains** the architecture and design of the Streamlit app
2. **Verifies** all dependencies and data files are in place
3. **Creates** `app.py` — the complete, self-contained Streamlit application
4. **Launches** the app directly from Jupyter

The app is the **portfolio centrepiece** of the project —  
a live, interactive demonstration of everything built across Notebooks 01–03.

---

## App Feature Overview

| Tab | What It Does |
|-----|-------------|
| 🔭 **Discover** | Natural-language search + rating history → hybrid recommendations |
| 📱 **App Detail** | Deep-dive into any app: ABSA radar, rating donut, similar apps |
| 💎 **Hidden Gems** | Sentiment-weighted discovery of underrated tools |
| 🗺 **Explore** | Full dataset overview: charts, category breakdowns, scatter plots |
| 📖 **About** | System architecture, formula, library credits |

---

## Self-Contained Design

The app works even if Notebook 02/03 outputs are not present.  
It detects what is available and falls back gracefully:

```
app_sentiment_scores.csv found?  →  Use full NLP scores (NB02)
app_sentiment_scores.csv missing?  →  Compute star-rating proxy automatically
item_similarity_matrix.npy found?  →  Load pre-computed matrix (NB03)
item_similarity_matrix.npy missing?  →  Rebuild from raw reviews at startup
```

All models are built once and **cached** via `@st.cache_resource`,  
so the 30-second startup only happens on first load.


## 📦 Step 1: Check Dependencies

In [1]:
import subprocess, sys, importlib, os

# Libraries the app requires
REQUIRED = {
    'streamlit':    'streamlit',
    'pandas':       'pandas',
    'numpy':        'numpy',
    'sklearn':      'scikit-learn',
    'matplotlib':   'matplotlib',
    'seaborn':      'seaborn',
    'scipy':        'scipy',
}

print("Checking required libraries...")
all_ok = True
for module, package in REQUIRED.items():
    try:
        mod = importlib.import_module(module)
        ver = getattr(mod, '__version__', 'installed')
        print(f"  ✅ {package:<20} {ver}")
    except ImportError:
        print(f"  ❌ {package:<20} NOT FOUND")
        all_ok = False

print()
if all_ok:
    print("✅ All dependencies satisfied — ready to run the app!")
else:
    print("❌ Some packages missing. Run the cell below to install them.")


Checking required libraries...
  ✅ streamlit            1.51.0
  ✅ pandas               2.3.3
  ✅ numpy                2.3.5
  ✅ scikit-learn         1.7.2
  ✅ matplotlib           3.10.6
  ✅ seaborn              0.13.2
  ✅ scipy                1.15.3

✅ All dependencies satisfied — ready to run the app!


In [2]:
# ── Install missing packages (only if needed) ────────────────────────────────
# Uncomment and run this cell if any package was missing above

# import subprocess, sys
# subprocess.run([sys.executable, '-m', 'pip', 'install',
#                 'streamlit', 'scikit-learn', 'matplotlib', 'seaborn',
#                 'scipy', 'pandas', 'numpy', '--break-system-packages', '-q'])
# print("✅ Installation complete — restart kernel, then re-run Step 1")

print("(Skip this cell if all libraries are already installed.)")


(Skip this cell if all libraries are already installed.)


## 📂 Step 2: Verify Data Files

In [3]:
import os

# ── Required (must exist) ─────────────────────────────────────────────────────
REQUIRED_FILES = [
    ('cleaned_reviews.csv',   'Raw reviews — CF matrix input'),
    ('cleaned_metadata.csv',  'App metadata — content + display'),
]

# ── Optional (improve quality if present) ─────────────────────────────────────
OPTIONAL_FILES = [
    ('app_sentiment_scores.csv',    'Full NLP sentiment scores (from NB02)'),
    ('review_sentiments.csv',       'Review-level sentiment details (from NB02)'),
    ('item_similarity_matrix.npy',  'Pre-computed CF similarity (from NB03)'),
    ('svd_app_factors.npy',         'SVD latent factors (from NB03)'),
    ('svd_sim_matrix.npy',          'SVD similarity matrix (from NB03)'),
    ('app_ids_ordered.json',        'App ID index mapping (from NB03)'),
]

print("=" * 62)
print("  FILE STATUS CHECK")
print("=" * 62)
print()
print("  REQUIRED files:")
all_required_ok = True
for fname, desc in REQUIRED_FILES:
    exists = os.path.exists(fname)
    size   = f"{os.path.getsize(fname)/1024:.0f} KB" if exists else "—"
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {fname:<38} {size:>8}   {desc}")
    if not exists:
        all_required_ok = False

print()
print("  OPTIONAL files (app will work without these):")
for fname, desc in OPTIONAL_FILES:
    exists = os.path.exists(fname)
    size   = f"{os.path.getsize(fname)/1024:.0f} KB" if exists else "—"
    status = "✅" if exists else "⚠️  using fallback"
    print(f"  {status}  {fname:<38} {size:>8}   {desc}")

print()
if all_required_ok:
    print("✅ All required files present — app is ready to launch!")
else:
    print("❌ Required files missing. Place them in the same folder as app.py.")


  FILE STATUS CHECK

  REQUIRED files:
  ✅  cleaned_reviews.csv                    25845 KB   Raw reviews — CF matrix input
  ✅  cleaned_metadata.csv                     617 KB   App metadata — content + display

  OPTIONAL files (app will work without these):
  ✅  app_sentiment_scores.csv                 699 KB   Full NLP sentiment scores (from NB02)
  ✅  review_sentiments.csv                  35156 KB   Review-level sentiment details (from NB02)
  ✅  item_similarity_matrix.npy              1815 KB   Pre-computed CF similarity (from NB03)
  ✅  svd_app_factors.npy                      188 KB   SVD latent factors (from NB03)
  ✅  svd_sim_matrix.npy                      1815 KB   SVD similarity matrix (from NB03)
  ✅  app_ids_ordered.json                      13 KB   App ID index mapping (from NB03)

✅ All required files present — app is ready to launch!


## 🏗️ Step 3: App Architecture Explained

Before launching, here is how the app is structured internally.  
Understanding this will help you **explain and defend** it in your viva.

### 1. Model Loading (`@st.cache_resource`)

```python
@st.cache_resource
def build_all_models():
    # Runs ONCE on first page load, then cached for the session
    # Builds: User-Item Matrix → Item-CF → SVD → Content TF-IDF → ABSA scores
```

`@st.cache_resource` is Streamlit's caching decorator for heavy objects.  
It means the 30-second model build happens **only once**, then every interaction is instant.

---

### 2. Hybrid Recommendation Function

```python
def hybrid_recommend(models, user_ratings, query_text, n, category,
                     alpha, beta, gamma, gem_boost):

    # Step 1: Build candidate pool (apply category/price filters)
    # Step 2: Compute CF scores  (item-item similarity × user ratings)
    # Step 3: Compute content scores (TF-IDF query similarity)
    # Step 4: Look up NLP sentiment scores (from app_sentiment_scores.csv)
    # Step 5: Fuse:  hybrid = α×CF + β×NLP + γ×content + gem_boost×is_gem
    # Step 6: Sort by hybrid_score, return top-N
```

---

### 3. Five UI Tabs

| Tab | Entry Point Function |
|-----|---------------------|
| Discover     | `tab_discover()`   |
| App Detail   | `tab_app_detail()` |
| Hidden Gems  | `tab_hidden_gems()`|
| Explore      | `tab_explore()`    |
| About        | `tab_about()`      |

---

### 4. ABSA at App Level

Because we aggregate reviews per app (up to 300 reviews each),  
the ABSA function processes one large text blob per app rather than individual reviews.  
This is computed during model loading and cached, so the UI shows instant results.


## ✍️ Step 4: Write `app.py` to Disk

In [4]:
# This cell reads the app source from the pre-written app.py
# (already in your folder). We verify it is syntactically correct.

import ast, os

app_path = 'app.py'

if not os.path.exists(app_path):
    print("❌ app.py not found in current directory.")
    print("   Make sure app.py is in the same folder as this notebook.")
else:
    with open(app_path, 'r', encoding='utf-8') as f:
        source = f.read()
    
    # Syntax check
    try:
        ast.parse(source)
        print(f"✅ app.py found and syntax is valid!")
        print(f"   File size : {os.path.getsize(app_path)/1024:.1f} KB")
        print(f"   Lines     : {len(source.splitlines())}")
        
        # Count functions
        tree = ast.parse(source)
        funcs = [n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]
        print(f"   Functions : {len(funcs)}")
        print(f"   Key functions: {', '.join(funcs[:8])}...")
    except SyntaxError as e:
        print(f"❌ Syntax error in app.py: {e}")


✅ app.py found and syntax is valid!
   File size : 62.2 KB
   Lines     : 1413
   Functions : 20
   Key functions: star_str, fmt_number, score_bar, clean_text_fn, build_all_models, hybrid_recommend, find_similar_apps, make_score_chart...


## 🧪 Step 5: Dry-Run — Test Models Without UI

In [5]:
# Run the model pipeline exactly as the app does, without launching Streamlit.
# This confirms everything works before opening the browser.

import pandas as pd, numpy as np, warnings, re, json
from collections import defaultdict
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix

warnings.filterwarnings('ignore')

print("🔄 Running dry-run model build (same logic as app startup)...")
print()

# ── Load data ─────────────────────────────────────────────────────────────────
reviews  = pd.read_csv('cleaned_reviews.csv')
metadata = pd.read_csv('cleaned_metadata.csv')
print(f"✅ Data loaded: {len(reviews):,} reviews, {len(metadata)} apps")

# ── Sentiment ─────────────────────────────────────────────────────────────────
try:
    sent_df = pd.read_csv('app_sentiment_scores.csv')
    metadata = metadata.merge(sent_df[['app_id','mean_sentiment_score','is_hidden_gem']],
                               on='app_id', how='left')
    print("✅ NLP sentiment scores loaded from app_sentiment_scores.csv")
except FileNotFoundError:
    scaler = MinMaxScaler(feature_range=(-1,1))
    metadata['mean_sentiment_score'] = scaler.fit_transform(
        metadata[['bayesian_avg']].fillna(metadata['bayesian_avg'].median())
    )
    s75 = metadata['bayesian_avg'].quantile(0.55)
    r25 = metadata['total_reviews'].quantile(0.30)
    metadata['is_hidden_gem'] = ((metadata['bayesian_avg'] >= s75) &
                                   (metadata['total_reviews'] <= r25)).astype(int)
    print(f"⚠️  Using star-rating proxy for sentiment ({metadata['is_hidden_gem'].sum()} gems found)")

metadata['mean_sentiment_score'] = metadata['mean_sentiment_score'].fillna(0)
metadata['is_hidden_gem']        = metadata['is_hidden_gem'].fillna(0).astype(int)

# ── User-Item Matrix ──────────────────────────────────────────────────────────
pivot = reviews.pivot_table(index='user_name', columns='app_id',
                             values='star_rating', aggfunc='mean')
user_means    = pivot.mean(axis=1)
pivot_centred = pivot.subtract(user_means, axis=0)
app_ids       = list(pivot.columns)
item_sparse   = csr_matrix(pivot_centred.fillna(0).values.T)
print(f"✅ User-Item matrix: {item_sparse.shape}")

# ── CF Similarity ─────────────────────────────────────────────────────────────
try:
    item_sim = np.load('item_similarity_matrix.npy')
    print("✅ Item-CF similarity loaded from file")
except FileNotFoundError:
    item_sim = cosine_similarity(item_sparse)
    np.fill_diagonal(item_sim, 0)
    print("✅ Item-CF similarity computed from scratch")

# ── SVD ───────────────────────────────────────────────────────────────────────
try:
    app_factors = np.load('svd_app_factors.npy')
    print("✅ SVD factors loaded from file")
except FileNotFoundError:
    svd = TruncatedSVD(n_components=50, random_state=42)
    app_factors = svd.fit_transform(item_sparse)
    print("✅ SVD factors computed from scratch")

# ── Content TF-IDF ────────────────────────────────────────────────────────────
metadata['content_text'] = (
    metadata['app_name'].fillna('') + ' ' +
    metadata['project_category'].fillna('') + ' ' +
    metadata['summary'].fillna('') + ' ' +
    metadata['description'].fillna('').str[:500]
).str.lower()

content_tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2),
                                 min_df=1, stop_words='english')
content_mat   = content_tfidf.fit_transform(metadata['content_text'])
content_sim   = cosine_similarity(content_mat)
np.fill_diagonal(content_sim, 0)
print(f"✅ Content TF-IDF matrix: {content_mat.shape}")

print()
print("=" * 50)
print("✅ ALL MODELS BUILT SUCCESSFULLY")
print("=" * 50)


🔄 Running dry-run model build (same logic as app startup)...

✅ Data loaded: 69,726 reviews, 482 apps
✅ NLP sentiment scores loaded from app_sentiment_scores.csv
✅ User-Item matrix: (482, 64415)
✅ Item-CF similarity loaded from file
✅ SVD factors loaded from file
✅ Content TF-IDF matrix: (482, 3000)

✅ ALL MODELS BUILT SUCCESSFULLY


In [6]:
# ── Test the hybrid recommender logic ─────────────────────────────────────────
print("🎯 Test 1: Top 5 recommendations (no history, no query)")
print("-" * 55)

app_to_idx    = {a: i for i, a in enumerate(app_ids)}
meta_app_idx  = {row['app_id']: i for i, row in metadata.iterrows()}
scaler2       = MinMaxScaler(feature_range=(0,1))
metadata['sentiment_norm'] = scaler2.fit_transform(metadata[['mean_sentiment_score']])
sentiment_lkp = dict(zip(metadata['app_id'], metadata['sentiment_norm']))
gem_lkp       = dict(zip(metadata['app_id'], metadata['is_hidden_gem']))

# Simple hybrid for test: bayesian_avg proxy CF + sentiment
def quick_hybrid(category=None, free_only=False, n=5, alpha=0.5, beta=0.35, gamma=0.15):
    cands = metadata.copy()
    if category: cands = cands[cands['project_category'] == category]
    if free_only: cands = cands[cands['free'] == True]
    
    results = []
    for _, row in cands.iterrows():
        aid   = row['app_id']
        cf_s  = float(row['bayesian_avg']) / 5.0 if pd.notna(row['bayesian_avg']) else 0.5
        nlp_s = sentiment_lkp.get(aid, 0.5)
        cont_s = 0.5
        gem    = gem_lkp.get(aid, 0)
        total = alpha + beta + gamma
        score = (alpha/total)*cf_s + (beta/total)*nlp_s + (gamma/total)*cont_s + 0.05*gem
        results.append({
            'app_name':    str(row['app_name'])[:35],
            'category':    row['project_category'],
            'avg_rating':  round(float(row['avg_rating']), 2) if pd.notna(row['avg_rating']) else 0,
            'hybrid_score':round(min(1.0, score), 4),
            'is_gem':      '💎' if gem else ''
        })
    
    df = pd.DataFrame(results).sort_values('hybrid_score', ascending=False).head(n)
    df.insert(0, 'rank', range(1, len(df)+1))
    return df

print(quick_hybrid(n=5).to_string(index=False))

print()
print("🎯 Test 2: Free Image Generation tools")
print("-" * 55)
print(quick_hybrid(category='Image Generation', free_only=True, n=5).to_string(index=False))

print()
print("🎯 Test 3: Hidden Gem hunt (β=0.70)")
print("-" * 55)
result3 = quick_hybrid(n=8, alpha=0.15, beta=0.70, gamma=0.15)
print(result3[result3['is_gem'] == '💎'].to_string(index=False) or "No gems in top 8 — try wider search")


🎯 Test 1: Top 5 recommendations (no history, no query)
-------------------------------------------------------
 rank                    app_name                   category  avg_rating  hybrid_score is_gem
    1 PDF AI: Document Summarizer            Productivity AI        4.60        0.9009      💎
    2           AI Code Generator       AI Coding Assistants        4.85        0.8970      💎
    3    AI Python Code Generator       AI Coding Assistants        4.21        0.8967      💎
    4  Email Writer: AI Generator Generative Text / Chatbots        4.81        0.8877      💎
    5      AI HTML Code Generator       AI Coding Assistants        4.64        0.8866      💎

🎯 Test 2: Free Image Generation tools
-------------------------------------------------------
 rank                       app_name         category  avg_rating  hybrid_score is_gem
    1  AR Drawing Paint Sketch Trace Image Generation        4.87        0.8401       
    2     AR Drawing & Anime Drawing Image Generation   

## 📊 Step 6: Generate Pre-Launch Report Charts

These charts are auto-generated here and also rendered inside the app's Explore tab.  
You can save them for your project report.


In [7]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sns.set_theme(style='dark', palette='muted')
plt.rcParams.update({'figure.dpi': 110})

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#0d1117')
fig.suptitle("Senti-Recommend — Pre-Launch Data Overview",
             fontsize=14, fontweight='bold', color='#e6edf3', y=0.98)

for ax in axes.flat:
    ax.set_facecolor('#161b22')
    ax.spines[['top','right','bottom','left']].set_color('#30363d')
    ax.tick_params(colors='#8b949e')

# 1. Apps per Category
cat_counts = metadata['project_category'].value_counts()
palette1 = ['#2dd4bf','#3b82f6','#a78bfa','#f59e0b','#fb7185']
axes[0,0].barh(cat_counts.index, cat_counts.values, color=palette1, edgecolor='none')
axes[0,0].set_title("Apps per Category", color='#e6edf3', fontweight='bold')
axes[0,0].set_xlabel("Count", color='#8b949e')
for i, (idx, val) in enumerate(cat_counts.items()):
    axes[0,0].text(val+0.3, i, str(val), va='center', color='#e6edf3', fontsize=9)

# 2. Rating Distribution
axes[0,1].hist(metadata['avg_rating'].dropna(), bins=30,
               color='#2dd4bf', edgecolor='#0d1117', alpha=0.85)
axes[0,1].set_title("App Rating Distribution", color='#e6edf3', fontweight='bold')
axes[0,1].set_xlabel("Average Rating", color='#8b949e')
axes[0,1].axvline(metadata['avg_rating'].mean(), color='#fbbf24',
                   linestyle='--', linewidth=1.5,
                   label=f"Mean: {metadata['avg_rating'].mean():.2f}")
axes[0,1].legend(fontsize=8, facecolor='#161b22', labelcolor='#e6edf3')

# 3. Sentiment Score Distribution
axes[0,2].hist(metadata['mean_sentiment_score'].dropna(), bins=30,
               color='#a78bfa', edgecolor='#0d1117', alpha=0.85)
axes[0,2].set_title("Sentiment Score Distribution", color='#e6edf3', fontweight='bold')
axes[0,2].set_xlabel("NLP Sentiment Score", color='#8b949e')
axes[0,2].axvline(0, color='red', linestyle='--', linewidth=1, alpha=0.5)

# 4. Ratings vs Reviews scatter
axes[1,0].scatter(metadata['total_reviews'].clip(upper=metadata['total_reviews'].quantile(0.95)),
                   metadata['avg_rating'],
                   c=metadata['is_hidden_gem'].map({0:'#3b82f6', 1:'#fbbf24'}),
                   alpha=0.55, s=40, edgecolors='none')
axes[1,0].set_title("Reviews vs Avg Rating", color='#e6edf3', fontweight='bold')
axes[1,0].set_xlabel("Total Reviews (95th pct cap)", color='#8b949e')
axes[1,0].set_ylabel("Avg Rating", color='#8b949e')
blue_patch  = mpatches.Patch(color='#3b82f6', label='Normal')
gold_patch  = mpatches.Patch(color='#fbbf24', label='Hidden Gem 💎')
axes[1,0].legend(handles=[blue_patch, gold_patch], fontsize=8,
                  facecolor='#161b22', labelcolor='#e6edf3')



# 5. Bayesian Average by Category
cat_order = metadata.groupby('project_category')['bayesian_avg'].mean().sort_values().index
data_box = [metadata[metadata['project_category']==c]['bayesian_avg'].dropna().values
            for c in cat_order]
bp = axes[1,1].boxplot(data_box, labels=[c.split('/')[0].strip()[:18] for c in cat_order],
                        patch_artist=True, medianprops={'color':'#0d1117','linewidth':2})
for patch, color in zip(bp['boxes'], ['#2dd4bf','#3b82f6','#a78bfa','#f59e0b','#fb7185']):
    patch.set_facecolor(color); patch.set_alpha(0.8)
axes[1,1].set_title("Bayesian Avg by Category", color='#e6edf3', fontweight='bold')
axes[1,1].tick_params(axis='x', rotation=15)

# 6. Hidden Gem Overview
gem_cats = metadata[metadata['is_hidden_gem']==1]['project_category'].value_counts()
non_cats = metadata[metadata['is_hidden_gem']==0]['project_category'].value_counts()
x_pos    = range(len(cat_counts))
w = 0.4
axes[1,2].bar([x - w/2 for x in x_pos],
               [non_cats.get(c,0) for c in cat_counts.index],
               w, label='Normal',      color='#3b82f6', edgecolor='none', alpha=0.8)
axes[1,2].bar([x + w/2 for x in x_pos],
               [gem_cats.get(c,0) for c in cat_counts.index],
               w, label='Hidden Gem',  color='#fbbf24', edgecolor='none', alpha=0.8)
axes[1,2].set_xticks(list(x_pos))
axes[1,2].set_xticklabels([c.split('/')[0].strip()[:12] for c in cat_counts.index],
                            rotation=20, ha='right', fontsize=8, color='#8b949e')
axes[1,2].set_title("Normal vs Hidden Gem per Category", color='#e6edf3', fontweight='bold')
axes[1,2].legend(fontsize=8, facecolor='#161b22', labelcolor='#e6edf3')

plt.tight_layout()
plt.savefig('prelaunch_dashboard.png', bbox_inches='tight', dpi=130, facecolor='#0d1117')
plt.show()
print("✅ Chart saved: prelaunch_dashboard.png")


✅ Chart saved: prelaunch_dashboard.png


## 🚀 Step 7: Launch the Streamlit App

In [8]:
import subprocess, os, sys, time

print("=" * 60)
print("  LAUNCHING SENTI-RECOMMEND")
print("=" * 60)
print()
print("Starting Streamlit server...")
print()

# Check app.py exists
if not os.path.exists('app.py'):
    print("❌ app.py not found. Run this cell from the same directory as app.py.")
else:
    # Launch in background
    process = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'app.py',
         '--server.headless', 'true',
         '--server.port', '8501',
         '--theme.base', 'dark'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    
    time.sleep(3)  # Give it a moment to start
    
    print("✅ Streamlit server started!")
    print()
    print("  🌐  Open your browser and go to:")
    print()
    print("       http://localhost:8501")
    print()
    print("  ── Or if running on a remote server/JupyterHub ──")
    print()
    print("       http://<your-server-ip>:8501")
    print()
    print("  The app takes ~30 seconds to build models on first load.")
    print("  After that, all interactions are instant.")
    print()
    print("  Press Ctrl+C or run the STOP cell below to shut down the server.")
    print()
    print(f"  Process ID: {process.pid}")
    
    # Store PID for later shutdown
    with open('.streamlit_pid', 'w') as f:
        f.write(str(process.pid))


  LAUNCHING SENTI-RECOMMEND

Starting Streamlit server...

✅ Streamlit server started!

  🌐  Open your browser and go to:

       http://localhost:8501

  ── Or if running on a remote server/JupyterHub ──

       http://<your-server-ip>:8501

  The app takes ~30 seconds to build models on first load.
  After that, all interactions are instant.

  Press Ctrl+C or run the STOP cell below to shut down the server.

  Process ID: 7280


In [9]:
# ── Alternative: Run from terminal (recommended for best experience) ──────────
print("ALTERNATIVE — Run from Terminal (Recommended):")
print()
print("  1. Open a terminal / Anaconda Prompt")
print("  2. Navigate to this folder:")
print(f"       cd {os.path.abspath('.')}")
print()
print("  3. Run:")
print("       streamlit run app.py")
print()
print("  4. The browser will open automatically at http://localhost:8501")


ALTERNATIVE — Run from Terminal (Recommended):

  1. Open a terminal / Anaconda Prompt
  2. Navigate to this folder:
       cd C:\Users\haris\DS\NED\Project PGD DSAI Spring 2026\sentirecommend\data\processed

  3. Run:
       streamlit run app.py

  4. The browser will open automatically at http://localhost:8501


In [10]:
# ── Stop the background server (run this cell to shut down) ──────────────────
import os, signal

if os.path.exists('.streamlit_pid'):
    with open('.streamlit_pid') as f:
        pid = int(f.read().strip())
    try:
        os.kill(pid, signal.SIGTERM)
        os.remove('.streamlit_pid')
        print(f"✅ Streamlit server (PID {pid}) stopped.")
    except ProcessLookupError:
        print("Server was already stopped.")
else:
    print("No running server found.")


✅ Streamlit server (PID 7280) stopped.


## 🎓 Step 8: Viva Preparation — How to Explain Each Tab

This section is specifically for your project defence.  
For each tab, here is a clear verbal explanation you can give.

---

### 🔭 Discover Tab — "The Main Engine"

> *"This tab is the primary user interface. A user can type a free-text query —  
> for example, 'easy image generator for a small business' — and the system  
> combines three signals. The collaborative filtering component looks at what  
> similar users have rated, the NLP component looks at actual review sentiment,  
> and the content-based component matches the query to app descriptions.  
> All three scores are fused using the weighted formula from my proposal."*

---

### 📱 App Detail Tab — "The ABSA Deep-Dive"

> *"Beyond a single star rating, this tab shows how users feel about five specific  
> aspects: ease of use, pricing, support, performance, and features. This is  
> Aspect-Based Sentiment Analysis — it detects which sentences in the reviews  
> mention each aspect, then measures whether the surrounding words are positive  
> or negative. The radar chart makes this immediately visual."*

---

### 💎 Hidden Gems Tab — "The Project's Unique Contribution"

> *"This is what separates my system from a simple popularity ranker.  
> A Hidden Gem is an app where users who have reviewed it genuinely love it —  
> high sentiment score — but it hasn't yet accumulated many reviews.  
> My system surfaces these by boosting the NLP weight and applying a gem bonus  
> in the hybrid formula. A standard recommender would bury these tools because  
> their total review count is low."*

---

### 🗺 Explore Tab — "Data Transparency"

> *"This tab shows the full dataset so the examiner can see the scale and  
> distribution of the data. The scatter plot shows the relationship between  
> star ratings and NLP sentiment scores — ideally these should correlate,  
> and you can see they do, with some interesting divergence cases."*

---

### 🎚 Weight Sliders — "Tunable Hybrid"

> *"The α, β, γ sliders let you tune the hybrid in real time.  
> α controls how much collaborative filtering influences results,  
> β controls NLP sentiment, and γ controls content similarity.  
> This allows the system to adapt to different use cases — for example,  
> a cold-start scenario where there is no user history should use higher γ."*


## ✅ Step 9: Project Completion Summary

In [11]:
import pandas as pd, os

print("=" * 65)
print("  SENTI-RECOMMEND — PROJECT COMPLETION SUMMARY")
print("=" * 65)
print()

notebooks = [
    ("NB01", "Data Acquisition & Cleaning",   "✅" if os.path.exists("cleaned_reviews.csv") else "⬜"),
    ("NB02", "NLP Sentiment Analysis",         "✅" if os.path.exists("app_sentiment_scores.csv") else "⚠️  (proxy active)"),
    ("NB03", "CF + Hybrid Recommender",        "✅" if os.path.exists("item_similarity_matrix.npy") else "⚠️  (rebuilt at runtime)"),
    ("NB04", "Streamlit Application",          "✅" if os.path.exists("app.py") else "❌"),
]

for nb, title, status in notebooks:
    print(f"  {status}  [{nb}]  {title}")

print()
print("  Deliverables:")
deliverables = [
    ("cleaned_metadata.csv",        "482 AI tools database"),
    ("cleaned_reviews.csv",         "69,726 user reviews"),
    ("app_sentiment_scores.csv",    "NLP sentiment per app"),
    ("item_similarity_matrix.npy",  "Item-CF model"),
    ("svd_app_factors.npy",         "SVD latent factors"),
    ("app.py",                      "Streamlit web application"),
]
for fname, desc in deliverables:
    exists = os.path.exists(fname)
    size   = f"{os.path.getsize(fname)/1024:.0f} KB" if exists else "—"
    print(f"  {'✅' if exists else '⚠️ '}  {fname:<38} {desc}  ({size})")

print()
metadata = pd.read_csv("cleaned_metadata.csv")
reviews  = pd.read_csv("cleaned_reviews.csv")

print("  Dataset Statistics:")
print(f"    AI tools indexed       : {len(metadata)}")
print(f"    User reviews analysed  : {len(reviews):,}")
print(f"    AI categories          : {metadata['project_category'].nunique()}")
print(f"    Unique users           : {reviews['user_name'].nunique():,}")
print(f"    Avg rating (global)    : {metadata['avg_rating'].mean():.2f}")

print()
print("=" * 65)
print("  🎓 PROJECT IS COMPLETE AND READY FOR SUBMISSION")
print("=" * 65)


  SENTI-RECOMMEND — PROJECT COMPLETION SUMMARY

  ✅  [NB01]  Data Acquisition & Cleaning
  ✅  [NB02]  NLP Sentiment Analysis
  ✅  [NB03]  CF + Hybrid Recommender
  ✅  [NB04]  Streamlit Application

  Deliverables:
  ✅  cleaned_metadata.csv                   482 AI tools database  (617 KB)
  ✅  cleaned_reviews.csv                    69,726 user reviews  (25845 KB)
  ✅  app_sentiment_scores.csv               NLP sentiment per app  (699 KB)
  ✅  item_similarity_matrix.npy             Item-CF model  (1815 KB)
  ✅  svd_app_factors.npy                    SVD latent factors  (188 KB)
  ✅  app.py                                 Streamlit web application  (62 KB)

  Dataset Statistics:
    AI tools indexed       : 482
    User reviews analysed  : 69,726
    AI categories          : 5
    Unique users           : 64,415
    Avg rating (global)    : 4.26

  🎓 PROJECT IS COMPLETE AND READY FOR SUBMISSION


## 🏁 Final Summary

---

### Project Notebook Chain

```
NB01_Data_Cleaning.ipynb
  └── outputs: cleaned_reviews.csv, cleaned_metadata.csv
        │
        ▼
NB02_NLP_Sentiment_Analysis.ipynb
  └── outputs: app_sentiment_scores.csv, review_sentiments.csv,
               nlp_model_metadata.json, [7 PNG charts]
        │
        ▼
NB03_CF_Hybrid_Recommender.ipynb
  └── outputs: item_similarity_matrix.npy, svd_app_factors.npy,
               svd_sim_matrix.npy, app_ids_ordered.json,
               cf_model_metadata.json, [8 PNG charts]
        │
        ▼
NB04_Streamlit_Application.ipynb  ←  YOU ARE HERE
  └── outputs: app.py  (the live interactive demo)
```

---

### What the App Demonstrates (Proposal Objectives)

| Proposal Objective | Where in App |
|-------------------|-------------|
| NLP User Satisfaction Scores | App Detail tab → ABSA radar + scores |
| Hybrid re-ranking algorithm | Discover tab → all results use α·CF + β·NLP + γ·Content |
| Hidden Gem identification | Hidden Gems tab + 💎 badges in all results |
| Cold-start mitigation | Query mode uses Content-Based fallback when no history given |
| Category/price filtering | Sidebar filters + Discover tab |
| Web-based dashboard | The entire Streamlit app |

---

> **To run the app:** `streamlit run app.py`  
> **To stop the app:** `Ctrl+C` in terminal  
> **Browser URL:** `http://localhost:8501`
